# Cài đặt các Thư viện sử dụng

In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from underthesea import word_tokenize
import os

In [2]:
import warnings
warnings.filterwarnings("ignore")

# Cài đặt hiển thị dataFrame

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Đọc dữ liệu


- vn_news_223_tdlfr.csv là file chứa news cần xử lý
- vietnamese-stopwords.txt chứa các stop word cần xử lý

In [4]:

file = os.path.join("../../Data_Train", "Vietnamese.csv")
data_df = pd.read_csv(file, encoding = 'utf-8')

file_stopword = os.path.join("../../Data_Train", "vietnamese-stopwords-dash.txt")
with open(file_stopword, 'r', encoding='utf-8') as file:
    stopwords = file.read().split('\n')

# Khám phá dữ liệu

## Mẫu dữ liệu

In [ ]:
data_df.sample(1)

## Xem thông tin

In [ ]:
data_df.info()

## Xem mô tả

In [ ]:
data_df.describe().round(1)

## Xem số dòng và số cột của dữ liệu

In [ ]:
num_rows=data_df.shape[0]
num_cols=data_df.shape[1]
print(num_rows)
print(num_cols)

## ý nghĩa của các dòng

### Kiểm tra dữ liệu có bị lặp ?

In [ ]:
dup=data_df.index.duplicated().sum()
dup

### ý nghĩa các cột

- text: nội dung của tin tức
- domain: đường dẫn đến trang web chứa tin tức 
- label: nhãn phân biệt tin giả hay tin thật

#### Cột có dtype là object nghĩa là sao?

- Trong Pandas, kiểu dữ liệu object thường ám chỉ chuỗi, nhưng thật ra kiểu dữ liệu object có thể chứa một đối tượng bất kỳ trong Python (vì thật ra ở bên dưới kiểu dữ liệu object chứa địa chỉ).
- Nếu một cột trong dataframe có dtype là object thì có thể các phần tử trong cột này sẽ có kiểu dữ liệu khác nhau
- Để biết được kiểu dữ liệu thật sự của các phần tử trong cột này thì ta phải truy xuất vào từng phần tử. Ta muốn xem thử trong nội bộ mỗi cột này có các kiểu dữ liệu nào

In [10]:
def open_object_dtype(s):
    dtypes = set()
    s=s.apply(type)
    dtypes.update(s.unique().tolist())
    return dtypes

In [ ]:
open_object_dtype(data_df['title'])

In [ ]:
open_object_dtype(data_df['text'])

In [ ]:
open_object_dtype(data_df['label'])

### Dữ liệu có bị thiếu không?

In [ ]:
data_df.isnull().sum()

#### Kiểm tra phân bố các class có chênh lệch không?

In [ ]:
# data quantity chart
def quantity_chart():
    counts_df1 = data_df['label'].value_counts()

    plt.figure(figsize=(6, 6))

    plt.pie(counts_df1, labels=counts_df1, autopct='%1.1f%%', startangle=90)
    plt.title('Bảng phân phối tin thật và tin giả')

    labels = ['Real' if label == 0 else 'Fake' for label in counts_df1.index]
    plt.legend(labels=labels, loc="best", fontsize=18)  # Tăng kích thước nhãn trong chú thích
    plt.tight_layout()
    plt.show()
quantity_chart()

#### Các thông tin thống kê

##### Chiều dài trung bình mỗi record là bao nhiêu?

In [ ]:
len_sum=0
for i in data_df['text']:
    len_sum+=len(i)
len_avg=len_sum/data_df['text'].count()
len_avg

##### Record dài nhất là bao nhiêu?

In [ ]:
max_len=0
for i in data_df['text']:
    if len(i)>max_len:
        max_len=len(i)
max_len

##### Record ngắn nhất là bao nhiêu?

In [ ]:
min_len=len(data_df['text'][0])
for i in data_df['text']:
    if len(i)<min_len:
        min_len=len(i)
min_len

### Tiền xử lí văn bản tiếng việt

#### Loại bỏ các đường link và các dấu câu, lowercase

In [19]:
from pyvi import ViTokenizer
from collections import Counter

In [20]:
def wordopt(text):
    text = text.lower()
    text = re.sub('https?:\/\/.*[\r\n]*', ' ', text)
    text = re.sub('[^\w\s]', ' ', text) 
    text = re.sub('\n', ' ', text)
    return text

#delete numbers
def delete_numbers(text):
    return re.sub(r'\d+', ' ', text)
#lower case
def lower(text):
    return text.lower()
#delete special characters
def remove_special_characters(text):
    ## Remove punctuations
    text = re.sub('[%s]' % re.escape("""!–"#$%&'()*+,،-./:;<=>؟?@[\]^`{|}~“”…"""), ' ', text)
    text = text.replace('؛',"", )
    text = re.sub('\s+', ' ', text)
    text =  " ".join(text.split())
    return text.strip()
#delete stop words
def remove_stopwords(text):
    # words = text.split()
    # mean_word = [word for word in words if word not in stopwords]
    # mean_text = " ".join(mean_word)
    clean_tokens = ' '.join([word for word in text.split() if word not in stopwords])
    return clean_tokens

def preprocess_nostop(text):
    text = delete_numbers(text)
    text = lower(text)
    text = remove_special_characters(text)
    return text


In [21]:
# Compound Vietnamese word
def tokenizerVN(text):
    return ViTokenizer.tokenize(text)
# Tokenizer
def tokenizer(text):
    return word_tokenize(text)
# Count token
def count_token(text):
    word = tokenizerVN(str(text))
    return len(word.split())
# Word Cloud
def top_count(dt):
    top = Counter([item for sublist in data_df[dt].apply(lambda x:str(x).split()) for item in sublist])
    temp = pd.DataFrame(top.most_common(50))
    temp.columns = ['Common_words','count']
    temp.style.background_gradient(cmap='Blues')
    return temp

In [22]:
data_df['article'] = ' |title| '+ data_df["title"] +' |text| '+  data_df["text"] 

In [ ]:
data_df.head(1)

In [24]:
data_df['Compound_Content'] = data_df['article'].apply(tokenizerVN)

In [25]:
data_df['Compound_Content_SW'] = data_df['Compound_Content'].apply(preprocess_nostop)

In [ ]:
data_df.head(1)

# **Visualization**

Visualize Functions

In [27]:
# word cloud

In [28]:
# from wordcloud import WordCloud
# import seaborn as sns

In [29]:
# # Top common words column
# def top_count_column(dt):
#     top = Counter([item for sublist in data_df[dt].apply(lambda x: str(x).split()) for item in sublist])
#     temp = pd.DataFrame(top.most_common(20))
#     temp.columns = ['Common_words', 'count']

#     # Visualization
#     plt.figure(figsize=(12, 8))
#     plt.barh(temp['Common_words'][::-1], temp['count'][::-1], color='skyblue')
#     plt.xlabel('Count')
#     plt.ylabel('Common Words')
#     plt.title('Top 20 Common Words')
#     plt.show()


# #Count word
# word_count_SW = data_df['Compound_Content_SW'].apply(count_token)


# # Plot data distribution
# def seaborn_token(word_count):
#     data_df['Word_count_SW'] = word_count
#     sns.stripplot(y='Word_count_SW', data = data_df , jitter=True)

#     plt.ylabel('Quantity Token')
#     plt.xlabel('Seaborn Strip Plot Count without stop words')

# def word_count():
#     plt.figure(figsize=(20, 7))  # Chỉ cần tạo một hình lớn

#     plt.hist(data_df["Word_count_SW"], bins=50)  # Vẽ biểu đồ histogram cho dữ liệu

#     plt.title('Word count have stop words')  # Đặt tiêu đề cho biểu đồ

#     plt.xlabel('Number of Words')  # Nhãn cho trục x (tùy chọn)
#     plt.ylabel('Frequency')  # Nhãn cho trục y (tùy chọn)

#     plt.show()  # Hiển thị biểu đồ


# def word_cloud(df):
#     plt.subplots(figsize=(20, 10))

#     wordcloud = WordCloud (
#                         background_color = 'white',
#                         width = 512,
#                         height = 384
#                             ).generate(' '.join(df))
#     plt.imshow(wordcloud) # image show
#     plt.axis('off') # to off the axis of x and y
#     # plt.savefig('Plotly_World_Cloud.png')
#     plt.show()

In [30]:
# data_df.head(1)

In [31]:
# top_count = top_count("Compound_Content_SW")
# top_count.style.background_gradient(cmap='Blues')

In [32]:
# top_count_column('Compound_Content_SW')

In [33]:
# seaborn_token(word_count_SW)

In [34]:
# word_count()

Word Cloud

In [35]:
# word_cloud(top_count['Common_words'])

In [36]:
#END

# Vector hóa

#### Tokenizer

In [37]:
def tokenize(sentence):
    return tokenizerVN(sentence).split()


### Mô hình hóa

- Chuyển đoạn văn tiếng Việt về vector, sử dụng CountVectorizer của sklearn
- với các tham số là danh sách stopwords tiếng Việt
- và tokenizer tách từ tiếng Việt

In [38]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib

In [39]:
X = data_df['Compound_Content_SW']
y = data_df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25,
                                                   random_state=30)

In [40]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

## CountVectorizer

In [41]:
model_vector_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_vectorizer_CV.joblib")
model_DTC_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_DTC_model_CV.joblib")
model_NB_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_NB_model_CV.joblib")
model_RFC_CV = os.path.join("../../Model/Vietnamese/CV", "Vietnamese_RFC_model_CV.joblib")


In [42]:
from sklearn.feature_extraction.text import CountVectorizer

In [43]:
vectorizerCV = CountVectorizer(
    stop_words = stopwords,
    tokenizer = tokenize,
)

In [50]:
Xv_trainCV = vectorizerCV.fit_transform(X_train)
Xv_testCV = vectorizerCV.transform(X_test)

In [ ]:
joblib.dump(vectorizerCV, model_vector_CV)

### DecisionTreeClassifier

In [ ]:
dtc_cv = DecisionTreeClassifier()
dtc_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_dt_cv = dtc_cv.predict(Xv_testCV)
print(dtc_cv.score(Xv_testCV, y_test))
print(classification_report(y_true=y_test, y_pred=pred_dt_cv))
#save model Decision Tree
joblib.dump(dtc_cv, model_DTC_CV)

### Navie bayes

In [ ]:
nb_cv = MultinomialNB()
nb_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_np_cv = nb_cv.predict(Xv_testCV)
print(nb_cv.score(Xv_testCV, y_test))
print(classification_report(y_true=y_test, y_pred=pred_np_cv))
#save model Naive Bayes
joblib.dump(nb_cv, model_NB_CV)

### RandomForest

In [57]:
rfc_cv = RandomForestClassifier()

In [ ]:
rfc_cv.fit(Xv_trainCV,y_train)

In [ ]:
pred_rfc_cv = rfc_cv.predict(Xv_testCV)
print(rfc_cv.score(Xv_testCV, y_test))
print(classification_report(y_true=y_test, y_pred=pred_rfc_cv))
#save model Naive Bayes
joblib.dump(rfc_cv, model_RFC_CV)

## TfidfVectorizer

In [60]:
model_vector_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_vectorizer_TF.joblib")
model_DTC_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_DTC_model_TF.joblib")
model_NB_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_NB_model_TF.joblib")
model_RFC_TF = os.path.join("../../Model/Vietnamese/TF", "Vietnamese_RFC_model_TF.joblib")


In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [62]:
vectorizerTF = TfidfVectorizer(stop_words=stopwords)

In [63]:
Xv_trainTF = vectorizerTF.fit_transform(X_train)
Xv_testTF = vectorizerTF.transform(X_test)

In [ ]:
joblib.dump(vectorizerTF, model_vector_TF)

### DecisionTreeClassifier

In [ ]:
dtc_tf = DecisionTreeClassifier()
dtc_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_dtc_tf = dtc_tf.predict(Xv_testTF)
print(dtc_tf.score(Xv_testTF, y_test))
print(classification_report(y_true=y_test, y_pred=pred_dtc_tf))
#save model Decision Tree
joblib.dump(dtc_tf, model_DTC_TF)

### Navie bayes

In [ ]:
nb_tf = MultinomialNB()
nb_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_nb_tf = nb_tf.predict(Xv_testTF)
print(nb_tf.score(Xv_testTF, y_test))
print(classification_report(y_true=y_test, y_pred=pred_nb_tf))
#save model Naive Bayes
joblib.dump(nb_tf, model_NB_TF)

### RandomForest

In [69]:
rfc_tf = RandomForestClassifier()

In [ ]:
rfc_tf.fit(Xv_trainTF,y_train)

In [ ]:
pred_rfc_tf = rfc_tf.predict(Xv_testTF)
print(rfc_tf.score(Xv_testTF, y_test))
print(classification_report(y_true=y_test, y_pred=pred_rfc_tf))
#save model Naive Bayes
joblib.dump(rfc_tf, model_RFC_TF)